### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="customer_satisfaction_in_airline",
    dataset_year="2023",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/yakhyojon/customer-satisfaction-in-airline",
    download_description="""
We download the data from Kaggle and uzip it to a predefined folder.

mkdir -p local-data-warehouse/customer_satisfaction_in_airline/ && cd local-data-warehouse/customer_satisfaction_in_airline/ && kaggle datasets download yakhyojon/customer-satisfaction-in-airline && cd ../../ && unzip local-data-warehouse/customer_satisfaction_in_airline/customer-satisfaction-in-airline.zip -d local-data-warehouse/customer_satisfaction_in_airline/ && rm local-data-warehouse/customer_satisfaction_in_airline/customer-satisfaction-in-airline.zip
""",
    # References
    academic_reference_bibtex="""@misc{yakhyojon2023airlinesatisfaction,
    author = {Kaggle User Yakhyojon},
    title = {Customer Satisfaction in Airline.},
    year = {2023},
    howpublished = {url{https://www.kaggle.com/datasets/yakhyojon/customer-satisfaction-in-airline}},
    note = {Kaggle}
}
""",
    academic_reference_bibtex_key="yakhyojon2023airlinesatisfaction",
    license="Public Domain",
    data_tags=["IID"],
    curation_comments="""
- We renamed the target column from "satisfaction" to "satisfied" for clarity and mapped the values to "Yes" and "No"
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="satisfied",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="satisfied",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/Invistico_Airline.csv", header=0)

feature_names = [
    "satisfaction",
    "Customer Type",
    "Age",
    "Type of Travel",
    "Class",
    "Flight Distance",
    "Seat comfort",
    "Departure/Arrival time convenient",
    "Food and drink",
    "Gate location",
    "Inflight wifi service",
    "Inflight entertainment",
    "Online support",
    "Ease of Online booking",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness",
    "Online boarding",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes"
]

df.rename(columns={"satisfaction": "satisfied"}, inplace=True)

cat_features = [
    "satisfied",
    "Customer Type",
    "Type of Travel",
    "Class",
    "Seat comfort",
    "Departure/Arrival time convenient",
    "Food and drink",
    "Gate location",
    "Inflight wifi service",
    "Inflight entertainment",
    "Online support",
    "Ease of Online booking",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness",
    "Online boarding",
]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df["satisfied"] = df["satisfied"].map({"satisfied": "Yes", "dissatisfied": "No"})

df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 129,880
Columns: 22
Use sampling: False (sample size: 129,880)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['Flight Distance', 'Arrival Delay in Minutes', 'Departure Delay in Minutes', 'Age', 'Seat comfort', 'Departure/Arrival time convenient', 'Leg room service', 'On-board service', 'Ease of Online booking', 'Online support']
Rows remaining as candidates after top-10 filter: 87 (of 129,880)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,satisfied,Customer Type,Age,Type of Travel,Class,Flight Distance,Seat comfort,Departure/Arrival time convenient,Food and drink,Gate location,Inflight wifi service,Inflight entertainment,Online support,Ease of Online booking,On-board service,Leg room service,Baggage handling,Checkin service,Cleanliness,Online boarding,Departure Delay in Minutes,Arrival Delay in Minutes
0,Yes,Loyal Customer,59,Business travel,Business,1470,4,4,4,4,5,4,4,4,4,4,4,5,4,3,7,0.0
1,No,disloyal Customer,22,Business travel,Eco,1771,1,1,1,4,4,1,5,4,3,4,3,1,4,4,0,0.0
2,Yes,Loyal Customer,55,Business travel,Business,3657,0,5,0,2,4,5,4,4,4,4,4,3,4,3,12,8.0
3,Yes,Loyal Customer,41,Business travel,Business,1796,0,4,0,1,2,4,5,3,3,3,3,5,3,3,0,0.0
4,No,Loyal Customer,42,Business travel,Eco,1709,2,3,3,3,2,2,2,2,4,4,4,1,3,2,0,0.0


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,satisfied,category,0.0,0.0,2.0,"Yes, No"
1,Customer Type,category,0.0,0.0,2.0,"Loyal Customer, disloyal Customer"
2,Type of Travel,category,0.0,0.0,2.0,"Business travel, Personal Travel"
3,Class,category,0.0,0.0,3.0,"Business, Eco, Eco Plus"
4,Seat comfort,category,0.0,0.0,6.0,"3, 2, 4, 1, 5, 0"
5,Departure/Arrival time convenient,category,0.0,0.0,6.0,"4, 5, 3, 2, 1, 0"
6,Food and drink,category,0.0,0.0,6.0,"3, 4, 2, 1, 5, 0"
7,Gate location,category,0.0,0.0,6.0,"3, 4, 2, 1, 5, 0"
8,Inflight wifi service,category,0.0,0.0,6.0,"4, 5, 3, 2, 1, 0"
9,Inflight entertainment,category,0.0,0.0,6.0,"4, 5, 3, 2, 1, 0"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
Age,129880.0,39.427957,15.119360,7.0,85.0
Flight Distance,129880.0,1981.409055,1027.115606,50.0,6951.0
Departure Delay in Minutes,129880.0,14.713713,38.071126,0.0,1592.0
Arrival Delay in Minutes,129487.0,15.091129,38.465650,0.0,1584.0


In [7]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column                            rank                                  
Baggage handling                  1                     4   48240  37.14
                                  2                     5   35748  27.52
                                  3                     3   24485  18.85
                                  4                     2   13432  10.34
                                  5                     1    7975   6.14
Checkin service                   1                     4   36481  28.09
                                  2                     3   35538  27.36
                                  3                     5   27005  20.79
                                  4                     2   15486  11.92
                                  5                     1   15369  11.83
Class                             1              Business   62160  47.86
                                  2                   Eco   58309  44.89
                                  3              Eco Plus    9411   7.25
Cleanliness                       1                     4   48795  37.57
                                  2                     5   35916  27.65
                                  3                     3   23984  18.47
                                  4                     2   13412  10.33
                                  5                     1    7768   5.98
Customer Type                     1        Loyal Customer  106100  81.69
                                  2     disloyal Customer   23780  18.31
Departure/Arrival time convenient 1                     4   29593  22.78
                                  2                     5   26817  20.65
                                  3                     3   23184  17.85
                                  4                     2   22794  17.55
                                  5                     1   20828  16.04
Ease of Online booking            1                     4   39920  30.74
                                  2                     5   34137  26.28
                                  3                     3   22418  17.26
                                  4                     2   19951  15.36
                                  5                     1   13436  10.34
Food and drink                    1                     3   28150  21.67
                                  2                     4   27216  20.95
                                  3                     2   27146  20.90
                                  4                     1   21076  16.23
                                  5                     5   20347  15.67
Gate location                     1                     3   33546  25.83
                                  2                     4   30088  23.17
                                  3                     2   24518  18.88
                                  4                     1   22565  17.37
                                  5                     5   19161  14.75
Inflight entertainment            1                     4   41879  32.24
                                  2                     5   29831  22.97
                                  3                     3   24200  18.63
                                  4                     2   19183  14.77
                                  5                     1   11809   9.09
Inflight wifi service             1                     4   31560  24.30
                                  2                     5   28830  22.20
                                  3                     3   27602  21.25
                                  4                     2   27045  20.82
                                  5                     1   14711  11.33
Leg room service                  1                     4   39698  30.57
                                  2                     5   34385  26.47
                                  3                     3   22467  17.30
                                  4  

In [8]:
# Target Distribution
target_df

,count,pct
satisfied,,
Yes,71087,54.73
No,58793,45.27


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to customer_satisfaction_in_airline/019d7367-4d56-7274-a470-1609f3a3d6fe


019d7367-4d56-7274-a470-1609f3a3d6fe
d76f335f7cc42c113e8b26d768e0a4818fd29f45cbed4ce7c6ca31031aa8d7f5
